### Imports

In [9]:
from langchain.document_loaders import YoutubeLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from dotenv import find_dotenv, load_dotenv
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
import textwrap

### Load environment variables and embeddings

In [10]:
load_dotenv(find_dotenv())
embeddings = OpenAIEmbeddings()

### Function for creating the database from the YouTube URL

In [11]:
def create_db_from_youtube_video_url(video_url):
    loader = YoutubeLoader.from_youtube_url(video_url)
    transcript = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)
    docs = text_splitter.split_documents(transcript)

    db = FAISS.from_documents(docs, embeddings)
    return db

### Function for getting the response from query

In [12]:
def get_response_from_query(db, query, k=4):
    docs = db.similarity_search(query, k=k)
    docs_page_content = " ".join([d.page_content for d in docs])

    chat = ChatOpenAI(model_name="gpt-3.5-turbo-16k", temperature=0.2)

    # Template to use for the system message prompt
    template = """
        You are a helpful assistant that that can answer questions about youtube videos 
        based on the video's transcript: {docs}
        
        Only use the factual information from the transcript to answer the question.
        
        If you feel like you don't have enough information to answer the question, say "I don't know".
        
        """

    system_message_prompt = SystemMessagePromptTemplate.from_template(template)

    # Human question prompt
    human_template = "Answer the following question: {question}"
    human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

    chat_prompt = ChatPromptTemplate.from_messages(
        [system_message_prompt, human_message_prompt]
    )

    chain = chat_prompt | chat

    response = chain.invoke({"question": query, "docs": docs_page_content}).content
    response = response.replace("\n", "")
    return response, docs

### Example usage

In [16]:
# Example usage 1:
video_url = "https://www.youtube.com/watch?v=3JW732GrMdg" # Video from FireShip about using PostgreSQL in webdev
db = create_db_from_youtube_video_url(video_url)

query = "what is this video about?"
response, docs = get_response_from_query(db, query)
print(textwrap.fill(response, width=100))

This video is about using PostgreSQL in web development and exploring various unorthodox and
unconventional ways to utilize PostgreSQL for different tasks. The video highlights the advantages
of PostgreSQL over other databases and showcases examples such as working with unstructured data
using JSON, implementing cron jobs, creating an in-memory cache, utilizing vector data types for AI
applications, and building a full-text search engine.


In [17]:
# Example usage 2:
video_url = "https://www.youtube.com/watch?v=_IOh0S_L3C4" # Video from Slidebean - AI Has a Fatal Flaw—And Nobody Can Fix It
db = create_db_from_youtube_video_url(video_url)

query = "what is AI flaw that the video is talking about?"
response, docs = get_response_from_query(db, query)
print(textwrap.fill(response, width=100))

The video is talking about the flaw in AI models, specifically the current machine learning
algorithms. The flaw is that there is a limit to the current models' capabilities, and they cannot
be trained to reach perfection or a very small error rate due to the lack of sufficient training
data.


In [18]:
# Example usage 3:
video_url = "https://www.youtube.com/watch?v=P-TANCVoHlc" # Video from Drew Gooden - Technology isn't fun anymore
db = create_db_from_youtube_video_url(video_url)

query = "why isn't technology fun anymore?"
response, docs = get_response_from_query(db, query)
print(textwrap.fill(response, width=100))

According to the transcript, technology isn't fun anymore because many technological advancements
are driven by motivations such as manipulation, surveillance, and profit rather than a genuine
desire to solve problems and improve people's lives. The speaker also mentions that advancements in
technology often feel predatory and manipulative, designed to be addictive rather than enjoyable
additions to one's life. Additionally, the speaker expresses frustration with the lack of noticeable
jumps in phone technology and the focus on charging people for things they already own rather than
expanding the experience. The speaker also mentions that technology is making people dumber and that
the current economic system encourages disposable and easily replaceable technology.
